# S2 cointegration — Monte Carlo EV vs SPY

Sealed OOS net returns only. **This notebook is EV vs SPY** (expected value, HAC/bootstrap significance of the mean, $P(\mathrm{not\ beat\ SPY})$). It does not compute prop-firm pass rates.


## 0. Imports & Config


In [ ]:
import os
import sys

import pandas as pd
from IPython.display import display

cur = os.path.abspath(os.getcwd())
ROOT = cur
for _ in range(12):
    if os.path.isfile(os.path.join(cur, "pyproject.toml")) and os.path.isdir(
        os.path.join(cur, "06_risk")
    ):
        ROOT = cur
        break
    parent = os.path.dirname(cur)
    if parent == cur:
        break
    cur = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from risk.analytics.monte_carlo.loaders import (
    aligned_strategy_spy,
    find_repo_root,
    load_sealed_s2,
)
from risk.analytics.monte_carlo.report import run_ev_vs_spy

ROOT = find_repo_root(ROOT)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("ROOT", ROOT)

SLEEVE = "s2"
BAR = "D"
PERIODS_PER_YEAR = 252
DEFAULT_H = 63
DEFAULT_BLOCK = 10.0
DEFAULT_N_SIM = 400
LEVERAGE = 1.0  # k on strategy simple returns only (not VT re-run)
HAIRCUT_BPS = 0.0  # subtract from each strategy return before scaling
N_BOOTSTRAP = 600  # §2 significance + §3 run_ev_vs_spy


## 1. Data Loading


In [ ]:
FRAME = aligned_strategy_spy(load_sealed_s2(ROOT), bar=BAR)
print(FRAME.tail())
print("n_bars", len(FRAME), "start", FRAME.index.min().date(), "end", FRAME.index.max().date())


## 2. Historical EV & significance

Here we look at the **historical mean** of sealed OOS period **returns**.
We use H0: $E[r]=0$ when evaluating the **t-stat and p-value**.
Bootstrap refers to $P^*(\hat\mu^*\le 0)$.
PSR is a secondary Sharpe quality — not an EV-significance metric.

Excess vs SPY significance is distinct from $P(\mathrm{not\ beat\ SPY})$ in §3–§5.


In [ ]:
from risk.analytics.monte_carlo.ev_stats import ev_significance, excess_returns
from risk.analytics.monte_carlo.plots import significance_frame

hist = ev_significance(
    FRAME["strategy"],
    periods_per_year=PERIODS_PER_YEAR,
    mean_block_length=DEFAULT_BLOCK,
    n_bootstrap=N_BOOTSTRAP,
    random_seed=0,
)
excess = ev_significance(
    excess_returns(FRAME["strategy"], FRAME["spy"]),
    periods_per_year=PERIODS_PER_YEAR,
    mean_block_length=DEFAULT_BLOCK,
    n_bootstrap=N_BOOTSTRAP,
    random_seed=0,
)
print("Strategy EV significance (historical mean)")
display(significance_frame(hist))
print("Excess vs SPY EV significance (distinct from P(not beat SPY))")
display(significance_frame(excess))


## 3. Joint bootstrap + MC run

Stationary block bootstrap draws **paired** strategy and SPY paths (same block indices).
Independent resampling would invalidate $P(\mathrm{not\ beat\ SPY})$.

`LEVERAGE` scales **strategy** simple returns only (`r' = k r`); it is a risk-budget overlay, not a re-run of sleeve vol-targeting.
`HAIRCUT_BPS` subtracts from each strategy return before scaling. Horizon is `DEFAULT_H` bars.

This section runs the simulation and prints the **headline scorecard** only; path plots follow in §4–§5.


In [ ]:
PACK = {}

pack = run_ev_vs_spy(
    FRAME,
    n_simulations=int(DEFAULT_N_SIM),
    horizon=int(DEFAULT_H),
    leverage=float(LEVERAGE),
    mean_block_length=float(DEFAULT_BLOCK),
    periods_per_year=PERIODS_PER_YEAR,
    random_seed=0,
    n_bootstrap=N_BOOTSTRAP,
    haircut_bps=float(HAIRCUT_BPS),
)
PACK.update(pack)
print("MC headline (no prop-firm pass rates)")
display(pack["headline"].to_frame("value"))


## 4. Pathwise holes

Not $P(\mathrm{ever\ underwater})$. Max-DD distribution, time spent below the peak, bars to recover from the trough, and the scatter of **terminal wealth vs max DD** (is $E[W_H]$ bought with a deep hole?). Top-decile EV share flags a fragile right tail.


In [ ]:
if not PACK:
    raise RuntimeError("run §3 first")
display(PACK["holes_summary"].to_frame("value"))
display(PACK["concentration"].to_frame("value"))
display(PACK["max_dd_hist"])
display(PACK["dd_scatter"])


## 5. Joint paths vs SPY

Same paired bootstrap columns as $P(\mathrm{not\ beat\ SPY})$. Median pathwise beta/corr, down-market capture (mean strategy return on bars with SPY $<0$), and $P(W_s\le W_{\mathrm{spy}}\mid W_{\mathrm{spy}}<1)$.

Gold/red lines on the fan are **sealed OOS** wealth over the last (and, if the sample is longer, first) $H$ bars. The excess fan is $W_{\mathrm{strat}}/W_{\mathrm{SPY}}$.


In [ ]:
if not PACK:
    raise RuntimeError("run §3 first")
display(PACK["joint_shape"].to_frame("value"))
display(PACK["fan"])
display(PACK["excess_fan"])
display(PACK["terminals"])
print("OOS terminal percentile among simulated paths", PACK["headline"]["oos_terminal_percentile"])


## 6. Evaluation

- **§2** — historical mean significance (strategy and excess vs SPY).
- **§3** — MC headline scorecard (`p_not_beat_spy`, horizon EV, CVaR).
- **§4** — pathwise hole risk (max DD, concentration).
- **§5** — joint path geometry vs SPY (fans, beta/corr, OOS terminal percentile).

S2 sealed series is **daily** net book returns, aligned to SPY close-to-close.


In [ ]:
# Optional HMM (univariate strategy only — not for P(beat SPY))
from risk.analytics.monte_carlo.hmm_simulator import GaussianHMMSimulator
from risk.analytics.monte_carlo.ev_stats import horizon_ev, cvar, terminal_simple_return

hmm = GaussianHMMSimulator(n_simulations=200, random_seed=0)
hmm.fit(FRAME["strategy"])
hmm_paths = hmm.simulate(DEFAULT_H)
print(hmm.summary(hmm_paths))
print("HMM horizon EV (strategy only)", horizon_ev(hmm_paths))
print("HMM CVaR5", cvar(terminal_simple_return(hmm_paths), alpha=0.05))
